# 🇨🇱 HipoRefi-CL: Demostración Interactiva de Optimización Hipotecaria

> **Motor Cuantitativo, Costos de Portabilidad (Ley 21.236) y Toma de Decisiones Financieras**  
> Este notebook permite validar y experimentar interactivamente con los módulos construidos en los **Hitos 1, 2 y 3**.

### Casos de Uso Demostrados:
1. **Extracción Documental:** Lectura y parsing de cartolas hipotecarias en PDF.
2. **Persistencia y Mercado:** Consulta de valor de la UF, TPM y tasas bancarias en DuckDB.
3. **Costos Operacionales:** Desglose legal de gastos de cambio (prepago Art. 100 LGB, arancel CBR con 50% descuento, exención D.L. 3475).
4. **Evaluación de Mercado:** Comparación del crédito actual contra 6 bancos chilenos y mutuarias, calculando **VPN** y **Payback**.
5. **Curvas de Amortización y Payback Dinámico:** Gráficos interactivos con Plotly.
6. **Detección de la Falacia del Dividendo:** Análisis cuantitativo de por qué bajar cuota alargando plazo suele ser un grave error patrimonial.

In [1]:
# Configuración de entorno y carga de módulos del proyecto
import sys
from pathlib import Path

# Añadir directorio raíz al path de Python
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from src.parsers.pdf_reader import PDFReader
from src.parsers.statement_extractor import StatementExtractor
from src.scrapers.market_service import MarketDataService
from src.core.amortizer import FrenchAmortizer, MortgageParams
from src.core.switching_costs import SwitchingCostCalculator
from src.core.metrics import RefinanceAnalyzer

print('✓ Entorno configurado correctamente. Módulos de HipoRefi-CL listos.')

✓ Entorno configurado correctamente. Módulos de HipoRefi-CL listos.


## 1. Extracción Documental de Cartola Hipotecaria (PDF)
Simulamos un comprobante bancario típico de **Banco de Chile** y lo procesamos con `StatementExtractor`.

In [2]:
# Cartola sintética de prueba
sample_pdf_lines = [
    'BANCO DE CHILE',
    'CARTOLA MENSUAL CRÉDITO HIPOTECARIO',
    'Operación N° 45892019-3',
    'Titular: Juan Pérez González',
    'Edad titular: 42 años',
    'Saldo de Capital Insoluto: 3.200,00 UF',
    'Tasa de Interés Anual Pactada: 5,20 %',
    'Dividendo N° 60 de 240 (180 cuotas pendientes)',
    'Dividendo Financiero: 21,50 UF',
    'Seguro Desgravamen: 0,90 UF',
    'Seguro Incendio y Sismo: 0,70 UF',
    'Dividendo Total a Pagar: 23,10 UF'
]

pdf_bytes = PDFReader.create_synthetic_pdf(sample_pdf_lines)
extractor = StatementExtractor(mode='auto')
statement = extractor.extract_from_pdf(pdf_bytes)

print(f'Banco Detectado       : {statement.bank_name}')
print(f'N° de Operación       : {statement.operation_number}')
print(f'Saldo Insoluto        : {statement.current_balance_uf:,.2f} UF')
print(f'Tasa Actual           : {statement.annual_interest_rate_pct:.2f}% anual')
print(f'Meses Restantes       : {statement.remaining_installments} meses ({statement.remaining_installments/12:.1f} años)')
print(f'Dividendo Total Actual: {statement.current_total_dividend_uf:.2f} UF')

Banco Detectado       : Banco de Chile
N° de Operación       : 45892019-3
Saldo Insoluto        : 3,200.00 UF
Tasa Actual           : 5.20% anual
Meses Restantes       : 180 meses (15.0 años)
Dividendo Total Actual: 23.10 UF


## 2. Consulta de Indicadores y Persistencia DuckDB
Consultamos el valor de la **UF** y las tasas de mercado promedio almacenadas en la base local.

In [3]:
market_service = MarketDataService()
uf_value = market_service.get_current_uf()
balance_clp = market_service.convert_uf_to_clp(statement.current_balance_uf)
dividend_clp = market_service.convert_uf_to_clp(statement.current_total_dividend_uf)

print(f'• Valor UF Actual en DuckDB : ${uf_value:,.2f} CLP')
print(f'• Saldo Deuda en Pesos      : ${balance_clp:,.0f} CLP')
print(f'• Dividendo en Pesos        : ${dividend_clp:,.0f} CLP / mes')

• Valor UF Actual en DuckDB : $40,942.74 CLP
• Saldo Deuda en Pesos      : $131,016,768 CLP
• Dividendo en Pesos        : $945,777 CLP / mes


## 3. Desglose de Gastos de Cierre bajo Ley N° 21.236
Calculamos los costos operacionales reales de migración considerando:
- **Comisión de Prepago:** Límite legal Art. 100 LGB (1.5 meses de intereses devengados).
- **Conservador de Bienes Raíces (CBR):** Arancel con 50% de descuento legal por subrogación.
- **Impuesto de Timbres y Estampillas (D.L. 3475):** 100% exento para refinanciamiento puro.

In [4]:
costs = SwitchingCostCalculator.calculate_total_costs(
    balance_uf=statement.current_balance_uf,
    current_annual_rate=statement.annual_interest_rate_pct / 100.0
)

costs_df = pd.DataFrame([
    {'Concepto': 'Comisión de Prepago (LGB Art. 100)', 'Monto_UF': costs.prepayment_penalty_uf},
    {'Concepto': 'Arancel CBR (50% Desc. Ley 21.236)', 'Monto_UF': costs.cbr_uf},
    {'Concepto': 'Tasación del Inmueble', 'Monto_UF': costs.appraisal_uf},
    {'Concepto': 'Estudio de Títulos', 'Monto_UF': costs.title_deed_uf},
    {'Concepto': 'Gastos Notariales Regulados', 'Monto_UF': costs.notary_uf},
    {'Concepto': 'Impuesto Timbres y Estampillas (D.L. 3475)', 'Monto_UF': costs.stamp_tax_uf},
])
costs_df['Monto_CLP'] = costs_df['Monto_UF'].apply(market_service.convert_uf_to_clp)

fig_costs = px.pie(
    costs_df, values='Monto_UF', names='Concepto',
    title=f'Composición de Gastos Operacionales (Total: {costs.total_cost_uf:.2f} UF / ${market_service.convert_uf_to_clp(costs.total_cost_uf):,.0f} CLP)',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_costs.show()
costs_df

,Concepto,Monto_UF,Monto_CLP
0,Comisión de Prepago (LGB Art. 100),20.320136,831962.043558
1,Arancel CBR (50% Desc. Ley 21.236),3.200000,131016.768000
2,Tasación del Inmueble,3.000000,122828.220000
3,Estudio de Títulos,4.000000,163770.960000
4,Gastos Notariales Regulados,1.000000,40942.740000
5,Impuesto Timbres y Estampillas (D.L. 3475),0.000000,0.000000


## 4. Comparación de Ofertas Bancarias y Valor Presente Neto (VPN)
Evaluamos cuantitativamente el refinanciamiento contra todas las ofertas del mercado chileno.

In [5]:
eval_result = market_service.evaluate_refinance_against_market(
    current_balance_uf=statement.current_balance_uf,
    current_annual_rate=statement.annual_interest_rate_pct / 100.0,
    months_remaining=statement.remaining_installments,
    current_total_dividend_uf=statement.current_total_dividend_uf
)

rows = []
for opp in eval_result['all_opportunities']:
    q = opp['bank_quote']
    e = opp['evaluation']
    rows.append({
        'Entidad': q['bank_name'],
        'Tasa_Anual': f"{q['annual_rate_pct']:.2f}%",
        'Dividendo_Nuevo_UF': q['monthly_total_dividend_uf'],
        'Ahorro_Mensual_UF': e['monthly_savings_uf'],
        'VPN_UF': e['npv_uf'],
        'Payback_Meses': e['payback_months'] if e['payback_months'] else 'No amortiza',
        'Dictamen': e['recommendation_flag']
    })

df_ranking = pd.DataFrame(rows)

# Gráfico de barras de VPN por banco
fig_npv = px.bar(
    df_ranking, x='Entidad', y='VPN_UF', color='Dictamen',
    title='Ganancia Patrimonial Neta (VPN en UF) por Entidad Financiera',
    text='VPN_UF',
    color_discrete_map={
        'RECOMENDADO': '#2ECC71',
        'EVALUAR_CON_CAUTELA': '#F39C12',
        'NO_CONVIENE': '#E74C3C'
    }
)
fig_npv.update_traces(texttemplate='%{text:.1f} UF', textposition='outside')
fig_npv.show()
df_ranking

,Entidad,Tasa_Anual,Dividendo_Nuevo_UF,Ahorro_Mensual_UF,VPN_UF,Payback_Meses,Dictamen
0,Mutuaria Security,4.23%,25.076,1.16,144.83,28,RECOMENDADO
1,Banco de Chile,4.38%,25.433,0.90,105.30,37,EVALUAR_CON_CAUTELA
2,BCI,4.40%,25.464,0.87,100.58,38,EVALUAR_CON_CAUTELA
3,Banco Santander,4.43%,25.507,0.80,89.29,42,EVALUAR_CON_CAUTELA
4,Scotiabank,4.44%,25.615,0.75,82.71,44,EVALUAR_CON_CAUTELA
5,Banco Itaú,4.48%,25.617,0.72,77.46,46,EVALUAR_CON_CAUTELA
6,BancoEstado,4.53%,25.631,0.64,65.62,52,EVALUAR_CON_CAUTELA


## 5. Curva de Retorno y Payback Descontado en el Tiempo
Visualizamos la trayectoria mes a mes del ahorro acumulado descontado de la mejor oferta frente a los gastos de cambio iniciales.

In [6]:
best = eval_result['best_opportunity']
best_quote = best['bank_quote']

# Generar tablas mes a mes para ambos créditos
params_curr = MortgageParams(
    principal=statement.current_balance_uf,
    annual_rate=statement.annual_interest_rate_pct / 100.0,
    months_remaining=statement.remaining_installments,
    fire_insurance_monthly_uf=0.70,
    life_insurance_rate_monthly=0.00028
)
sched_curr = FrenchAmortizer.generate_schedule(params_curr)

params_new = MortgageParams(
    principal=statement.current_balance_uf,
    annual_rate=best_quote['annual_rate_pct'] / 100.0,
    months_remaining=statement.remaining_installments,
    fire_insurance_monthly_uf=best_quote['fire_insurance_uf'],
    life_insurance_rate_monthly=0.00028
)
sched_new = FrenchAmortizer.generate_schedule(params_new)

months_axis = list(range(1, statement.remaining_installments + 1))
discount_rate = 0.025 / 12.0  # Mensual aproximado

monthly_diffs = [c['total_dividend_uf'] - n['total_dividend_uf'] for c, n in zip(sched_curr, sched_new)]
discounted_diffs = [diff / ((1.0 + discount_rate) ** m) for m, diff in enumerate(monthly_diffs, 1)]

# Flujo neto acumulado partiendo desde -G_k
cum_net_cashflow = []
running = -costs.total_cost_uf
for d in discounted_diffs:
    running += d
    cum_net_cashflow.append(running)

fig_payback = go.Figure()
fig_payback.add_trace(go.Scatter(
    x=months_axis, y=cum_net_cashflow,
    mode='lines',
    name='Flujo Acumulado Descontado (UF)',
    line=dict(color='#2980B9', width=3)
))
fig_payback.add_hline(
    y=0, line_dash='dash', line_color='red',
    annotation_text='Punto de Equilibrio (Break-Even)', annotation_position='top left'
)
payback_m = best['evaluation']['payback_months']
if payback_m:
    fig_payback.add_vline(
        x=payback_m, line_dash='dot', line_color='green',
        annotation_text=f'Payback: Mes {payback_m}', annotation_position='bottom right'
    )

fig_payback.update_layout(
    title=f'Trayectoria de Retorno de Inversión ({best_quote["bank_name"]})',
    xaxis_title='Mes',
    yaxis_title='Flujo Neto Acumulado Descontado (UF)',
    hovermode='x unified'
)
fig_payback.show()

## 6. Detección de 'La Falacia del Dividendo'
Muchos deudores cometen el error de renegociar su crédito alargando el plazo restante para obtener una cuota mensual más baja.
Aquí modelamos una oferta trampa: **Alargar de 15 años (180 meses) a 25 años (300 meses)** con una tasa ligeramente menor.

In [7]:
# Crédito alargado a 25 años (300 meses) con tasa atractiva al 4.80%
params_trap = MortgageParams(
    principal=statement.current_balance_uf,
    annual_rate=0.048,
    months_remaining=300,
    fire_insurance_monthly_uf=0.70,
    life_insurance_rate_monthly=0.00028
)
sched_trap = FrenchAmortizer.generate_schedule(params_trap)

trap_eval = RefinanceAnalyzer.evaluate(
    current_schedule=sched_curr,
    new_schedule=sched_trap,
    upfront_costs_uf=costs.total_cost_uf
)

div_orig = sched_curr[0]['total_dividend_uf']
div_trap = sched_trap[0]['total_dividend_uf']
total_paid_orig = sum(c['total_dividend_uf'] for c in sched_curr)
total_paid_trap = sum(c['total_dividend_uf'] for c in sched_trap) + costs.total_cost_uf

print(f'• Dividendo Actual (15 años)          : {div_orig:.2f} UF/mes')
print(f'• Dividendo Oferta Trampa (25 años)   : {div_trap:.2f} UF/mes  (¡Ahorro aparente de {div_orig - div_trap:.2f} UF/mes!)')
print(f'• Costo Total Original de la Deuda    : {total_paid_orig:,.1f} UF')
print(f'• Costo Total con Oferta Trampa       : {total_paid_trap:,.1f} UF (¡Paga {total_paid_trap - total_paid_orig:,.1f} UF MÁS en total!)')
print(f'• Destrucción Patrimonial Neta (VPN)  : {trap_eval.npv_uf:,.2f} UF (${market_service.convert_uf_to_clp(trap_eval.npv_uf):,.0f} CLP)')
print(f'• Dictamen Patrimonial                : [{trap_eval.recommendation_flag}]')

# Gráfico comparativo de desembolso total nominal
df_trap = pd.DataFrame([
    {'Escenario': 'Mantener Crédito Actual (15 años)', 'Desembolso_Total_UF': total_paid_orig},
    {'Escenario': 'Oferta Trampa: Alargar a 25 años', 'Desembolso_Total_UF': total_paid_trap},
])
fig_trap = px.bar(
    df_trap, x='Escenario', y='Desembolso_Total_UF', color='Escenario',
    title='Comparación de Destrucción Patrimonial: Crédito Actual vs Alargamiento de Plazo',
    text='Desembolso_Total_UF',
    color_discrete_sequence=['#3498DB', '#E74C3C']
)
fig_trap.update_traces(texttemplate='%{text:,.1f} UF', textposition='outside')
fig_trap.show()

• Dividendo Actual (15 años)          : 27.04 UF/mes
• Dividendo Oferta Trampa (25 años)   : 19.74 UF/mes  (¡Ahorro aparente de 7.29 UF/mes!)
• Costo Total Original de la Deuda    : 4,796.2 UF
• Costo Total con Oferta Trampa       : 5,846.2 UF (¡Paga 1,050.0 UF MÁS en total!)
• Destrucción Patrimonial Neta (VPN)  : -367.41 UF ($-15,042,772 CLP)
• Dictamen Patrimonial                : [NO_CONVIENE]
